In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT: Path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from eval_utils import LABEL_COLS, score_predictions

data: pd.DataFrame = pd.read_csv(ROOT / "data/processed/sbdh_splits.csv")
train: pd.DataFrame = data[data["split"] == "train"].reset_index(drop=True)
dev: pd.DataFrame = data[data["split"] == "dev"].reset_index(drop=True)
print(f"train: {len(train):,} | dev: {len(dev):,}")

# majority class per label, learned from train, applied to dev
majority: dict[str, int] = {col: int(train[col].mode()[0]) for col in LABEL_COLS}
dev_pred_majority: pd.DataFrame = pd.DataFrame({col: [majority[col]] * len(dev) for col in LABEL_COLS})

scores_majority: pd.DataFrame = score_predictions(dev[LABEL_COLS], dev_pred_majority, "majority")
print(majority)
print(scores_majority[["label", "macro_f1", "weighted_f1"]].round(3).to_string(index=False))

train: 4,803 | dev: 1,029
{'sdoh_community_present': 1, 'sdoh_community_absent': 0, 'sdoh_education': 0, 'sdoh_economics': 0, 'sdoh_environment': 1, 'behavior_alcohol': 3, 'behavior_tobacco': 3, 'behavior_drug': 0}
                 label  macro_f1  weighted_f1
sdoh_community_present     0.393        0.510
 sdoh_community_absent     0.467        0.818
        sdoh_education     0.493        0.956
        sdoh_economics     0.251        0.454
      sdoh_environment     0.259        0.493
      behavior_alcohol     0.107        0.194
      behavior_tobacco     0.099        0.163
         behavior_drug     0.155        0.493


In [2]:
import importlib, regex_baseline
importlib.reload(regex_baseline)
from eval_utils import per_class_report

SUB_COLS: list[str] = list(regex_baseline.SUBSTANCE_TERMS)
dev_pred_sub: pd.DataFrame = regex_baseline.predict_substances(dev["SOCIAL_HISTORY_TEXT"])

for col in SUB_COLS:
    print(f"\n=== {col} ===")
    print(per_class_report(dev[col], dev_pred_sub[col]))


=== behavior_alcohol ===
              precision    recall  f1-score   support

           0       0.90      0.70      0.79       267
           1       0.63      0.82      0.72       288
           2       0.40      0.43      0.41        56
           3       0.76      0.79      0.78       374
           4       0.00      0.00      0.00        44

    accuracy                           0.72      1029
   macro avg       0.54      0.55      0.54      1029
weighted avg       0.71      0.72      0.71      1029


=== behavior_tobacco ===
              precision    recall  f1-score   support

           0       0.89      0.85      0.87       197
           1       0.39      0.83      0.53       151
           2       0.81      0.60      0.69       281
           3       0.85      0.78      0.82       339
           4       0.00      0.00      0.00        61

    accuracy                           0.71      1029
   macro avg       0.59      0.61      0.58      1029
weighted avg       0.73  

In [3]:
from sklearn.metrics import confusion_matrix
import re

def show_errors(label: str, true_cls: int, pred_cls: int, n: int = 15) -> None:
    pattern: str = regex_baseline.SUBSTANCE_TERMS[label]
    mask = (dev[label] == true_cls) & (dev_pred_sub[label] == pred_cls)
    print(f"{label}: true={true_cls} pred={pred_cls} -> {int(mask.sum())} notes\n")
    for text in dev.loc[mask, "SOCIAL_HISTORY_TEXT"].head(n):
        hits = [s.strip() for s in re.split(regex_baseline.SPLIT, text) if re.search(pattern, s, flags=re.IGNORECASE)]
        print(" | ".join(hits)[:300])
        print("-" * 60)

for col in SUB_COLS:
    print(col)
    print(confusion_matrix(dev[col], dev_pred_sub[col], labels=[0, 1, 2, 3, 4]))

# largest drug error group: labeled never/none, predicted present
show_errors("behavior_drug", true_cls=3, pred_cls=1)

behavior_alcohol
[[188  44   8  27   0]
 [  5 237  11  35   0]
 [  0  19  24  13   0]
 [ 14  51  13 296   0]
 [  1  23   4  16   0]]
behavior_tobacco
[[167  13   8   9   0]
 [  3 125  10  13   0]
 [  2 103 170   6   0]
 [ 13  56   5 265   0]
 [  2  26  16  17   0]]
behavior_drug
[[556  23   2  72   0]
 [  3  21   1   4   0]
 [  0  12   8   4   1]
 [  0  67   2 231   1]
 [  0  13   0   8   0]]
behavior_drug: true=3 pred=1 -> 67 notes

drug use
------------------------------------------------------------
and illict drugs
------------------------------------------------------------
history of drug use
------------------------------------------------------------
drugs
------------------------------------------------------------
No history of tobacco, alcohol, or illicit | drug use
------------------------------------------------------------
or drug use
------------------------------------------------------------
cigs, drank ETOH or used recreational drugs
----------------------------------